# 17 - Upload data and model to Blob

Use the existing private container. This step adds Storage Blob Data Contributor for your signed-in user on this project storage account if needed. It uploads seven pinned files and downloads each to check SHA-256. No training VM or app replica is created. Storage operations and retained files can incur charges.


In [3]:
from pathlib import Path
import json
import platform
import subprocess
import sys

assert platform.system() == "Linux", "Use the WSL kernel."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
assert Path(sys.prefix).name == "mlop-pj", "Use the project environment."
status = json.loads((ROOT / "reports/azure-platform-deploy.json").read_text())
assert status["platform_verified"] and not status["errors"], "Finish 16 first."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from cloudlayer.blob_sync import files_for
files = files_for(ROOT)
print("Files:", len(files), "Total bytes:", sum(row["bytes"] for row in files))
for row in files:
    print(row["file"], row["sha256"][:12])


Files: 7 Total bytes: 1756369
data/raw/google_review_ratings.csv c07e7751f429
models/artifacts/abfd88e893ca/bundle.joblib 71f3ba41b4b9
data/processed/profiles.csv dd202f838ece
configs/data.json 2fe7512532fa
reports/data-source.json d706e73e4397
reports/bundle-manifest.json 1d9bd3942842
data/raw.dvc c4daedad2c07


## Allow your user to access Blob

Run this cell once. It grants only the Blob data role on the project storage account if missing, then checks data access. Role propagation may take a few minutes. Expect errors: [] and blob_access_ready: true. If it reports propagation, wait a few minutes and rerun this cell.


In [4]:
result = subprocess.run([sys.executable, "-m", "cloudlayer.blob_sync", "access"],
                        cwd=ROOT, capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr or "Blob access script failed.")
access = json.loads(result.stdout)
print(json.dumps(access, indent=2))
assert not access["errors"] and access["blob_access_ready"], "Stop and send the access report."


{
  "checked_at": "2026-10-11T15:35:15.387876+00:00",
  "phase": "access",
  "errors": [],
  "blob_access_ready": true,
  "resource_group": "rg-mlop-pj-492ea24d",
  "region": "centralindia",
  "suffix": "492ea24d",
  "project_tag": "mlop-pj-6688012",
  "budget_thb_before_credits": 150,
  "user_role_created": true,
  "role_scope": "project storage account"
}


## Upload and check

Run after access passes. Files use SHA-256 in their Blob names. Existing blobs are downloaded and checked; they are not overwritten. Expect seven files with download_sha256_matches: true, errors: [], and all_hashes_match: true.


In [5]:
result = subprocess.run([sys.executable, "-m", "cloudlayer.blob_sync", "upload"],
                        cwd=ROOT, capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr or "Blob upload script failed.")
report = json.loads(result.stdout)
print(json.dumps(report, indent=2))
assert not report["errors"] and report["all_hashes_match"], "Stop and send the upload report."


{
  "checked_at": "2026-10-11T15:35:40.681721+00:00",
  "phase": "upload",
  "errors": [],
  "data_uploaded": true,
  "all_hashes_match": true,
  "resource_group": "rg-mlop-pj-492ea24d",
  "region": "centralindia",
  "suffix": "492ea24d",
  "project_tag": "mlop-pj-6688012",
  "budget_thb_before_credits": 150,
  "files": [
    {
      "file": "data/raw/google_review_ratings.csv",
      "bytes": 637001,
      "sha256": "c07e7751f429ebe6225a77a32e61770dbb7197fd253f667e0434424adbaf7a33",
      "blob": "snapshots/c07e7751f429ebe6225a77a32e61770dbb7197fd253f667e0434424adbaf7a33/google_review_ratings.csv",
      "uploaded_now": true,
      "download_sha256_matches": true
    },
    {
      "file": "models/artifacts/abfd88e893ca/bundle.joblib",
      "bytes": 20399,
      "sha256": "71f3ba41b4b964f5beb31e3d6e13b835b1dd2538aa663741d8837301be9dff13",
      "blob": "snapshots/71f3ba41b4b964f5beb31e3d6e13b835b1dd2538aa663741d8837301be9dff13/bundle.joblib",
      "uploaded_now": true,
      "downlo

## Save and continue

Save the notebook. Reports: reports/azure-blob-access.json and reports/azure-blob-upload.json. Send the final result. This proves a Blob round trip for the current files. DVC remote push/pull and Azure ML training/registry are still next.

[Blob login authentication](https://learn.microsoft.com/en-us/azure/storage/blobs/authorize-data-operations-cli)
